In [ ]:
import os
os.chdir(r"..\models\..")

import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import pickle
from torch.utils.data import Dataset

from transformer.tokenizer import Tokenizer
from language_model.text_generator import TextGenerator

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

In [2]:
# Training Tokenizer
data = "data\\my_essays_data.txt"

with open(data, "r", encoding="utf-8") as f:
    text = f.read()

tokenizer = Tokenizer(vocab_size=100_000)
tokenizer.fit([text])

vocab_size = len(tokenizer.word_to_idx)
#print(f"Vocabulary Size: {vocab_size}")

with open("models/generation_tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [3]:
tokens = tokenizer.transform(text)
print(len(text))
print("Token count:", len(tokens))
print(tokens[:10])

print("Text type:", type(text))
print("Token count:", len(tokens))
print("Vocab size:", len(tokenizer.word_to_idx))

print(tokenizer.word_to_idx)


110846
Token count: 19249
[93, 1091, 1092, 1671, 15, 1093, 5, 190, 210, 9]
Text type: <class 'str'>
Token count: 19249
Vocab size: 3442
{'<PAD>': 0, '<UNK>': 1, 'the': 2, 'and': 3, 'to': 4, 'of': 5, 'in': 6, 'a': 7, 'it': 8, 'is': 9, 'that': 10, 'i': 11, 'for': 12, 'as': 13, 'was': 14, 's': 15, 'with': 16, 'this': 17, 'not': 18, 'also': 19, 'we': 20, 'be': 21, 'he': 22, 'or': 23, 'but': 24, 'are': 25, 'my': 26, 'how': 27, 'by': 28, 'who': 29, 'his': 30, 'on': 31, 'have': 32, 'more': 33, 'from': 34, 'people': 35, 'about': 36, 'one': 37, 'at': 38, 'can': 39, 'their': 40, 'which': 41, 'you': 42, 'what': 43, 'they': 44, 'such': 45, 'an': 46, 'these': 47, 'where': 48, 'new': 49, 't': 50, 'life': 51, 'if': 52, 'had': 53, 'when': 54, 'like': 55, 'time': 56, 'some': 57, 'only': 58, 'has': 59, 'all': 60, 'her': 61, 'being': 62, 'first': 63, 'other': 64, 'even': 65, 'its': 66, 'than': 67, 'there': 68, 'chapman': 69, 'will': 70, 'so': 71, 'me': 72, 'mrs': 73, 'city': 74, 'because': 75, 'were': 76

In [4]:
class TextDataset(Dataset):
    def __init__(self, tokens, seq_len):
        self.tokens = tokens
        self.seq_len = seq_len
        self.stride = seq_len // 2

        assert len(tokens) > seq_len, (
            f"Dataset too small! Tokens={len(tokens)}, seq_len={seq_len}"
        )

    def __len__(self):
        return (len(self.tokens) - self.seq_len) // self.stride
    
    def __getitem__(self, idx):

        start = idx * self.stride

        x = self.tokens[start:start + self.seq_len]
        y = self.tokens[start + 1:start + self.seq_len + 1]
       
        return torch.tensor(x), torch.tensor(y)

In [ ]:
# Training the model
dataset = TextDataset(tokenizer.transform(text), seq_len=100)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TextGenerator(
    vocab_size=vocab_size,
    embedding_dim=128,
    num_layers=3,
    num_heads=4,
    d_ff=512,
    max_len=128,
    dropout=0.1,
    device=device
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=3e-4)
batch_size = 64
epochs = 20

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x, y in dataloader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        output = model(x)

        loss = criterion(output.reshape(-1, output.shape[-1]), y.reshape(-1))
        loss.backward()

        # torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss / len(dataloader):.4f}")

Epoch 1 | Loss: 8.1188
Epoch 2 | Loss: 7.7013
Epoch 3 | Loss: 7.4082
Epoch 4 | Loss: 7.1984
Epoch 5 | Loss: 7.0237
Epoch 6 | Loss: 6.8739
Epoch 7 | Loss: 6.7502
Epoch 8 | Loss: 6.6559
Epoch 9 | Loss: 6.5843
Epoch 10 | Loss: 6.5342
Epoch 11 | Loss: 6.4976
Epoch 12 | Loss: 6.4746
Epoch 13 | Loss: 6.4549
Epoch 14 | Loss: 6.4356
Epoch 15 | Loss: 6.4147
Epoch 16 | Loss: 6.3909
Epoch 17 | Loss: 6.3655
Epoch 18 | Loss: 6.3371
Epoch 19 | Loss: 6.3086
Epoch 20 | Loss: 6.2754
